In [ ]:
# Block 1: Install or upgrade DuckDB in the current Jupyter environment.
import sys

!{sys.executable} -m pip install -U duckdb

In [ ]:
# Block 2: Import the Python libraries used throughout the notebook.
from pathlib import Path
import gzip
import duckdb
import pandas as pd
# '/home/jovyan/notebook-antigo/Vitória/Segmentação/Notas-fiscais.csv' privado/imdb/name.basics.tsv

In [ ]:
# Block 3: Define input and database paths, open the DuckDB connection, and configure basic execution settings.
# Set the folder that contains the IMDb .tsv files
DATA_DIR = Path("/home/jovyan/privado/imdb/")

DB_PATH = DATA_DIR / "imdb_sf.duckdb"

con = duckdb.connect(str(DB_PATH))

# Adjust these execution settings to match your machine
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='16GB';")

In [ ]:
# Block 4: Read the header row of a TSV or TSV.GZ file so the column names can be passed explicitly to DuckDB.
def read_tsv_header(file_path):
    file_path = Path(file_path)
    
    if file_path.suffix == ".gz":
        with gzip.open(file_path, "rt", encoding="utf-8", newline="") as f:
            header = f.readline().rstrip("\n\r")
    else:
        with open(file_path, "r", encoding="utf-8", newline="") as f:
            header = f.readline().rstrip("\n\r")
    
    cols = header.split("\t")
    return cols

In [ ]:
# Block 5: List the seven IMDb files that will be used in the pipeline.
FILES = {
    "name_basics": "name.basics.tsv",
    "title_akas": "title.akas.tsv",
    "title_basics": "title.basics.tsv",
    "title_crew": "title.crew.tsv",
    "title_episode": "title.episode.tsv",
    "title_principals": "title.principals.tsv",
    "title_ratings": "title.ratings.tsv",
}

In [ ]:
# Block 6: Create a DuckDB view over an IMDb TSV file using explicit column names and parser options that are robust for large TSV files.
def create_tsv_view_generic(con, view_name, file_path):
    file_path = Path(file_path)
    safe_path = str(file_path).replace("\\", "/").replace("'", "''")

    cols = read_tsv_header(file_path)

    col_parts = []
    for c in cols:
        c_escaped = c.replace("'", "''")
        col_parts.append("'" + c_escaped + "': 'VARCHAR'")

    columns_spec = "{" + ", ".join(col_parts) + "}"

    sql = f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT *
    FROM read_csv(
        '{safe_path}',
        delim='\\t',
        header=true,
        columns={columns_spec},
        auto_detect=false,
        nullstr='\\N',
        compression='auto',
        strict_mode=false,
        null_padding=true,
        parallel=false,
        quote='',
        escape='',
        max_line_size=50000000
    );
    """
    con.execute(sql)

In [ ]:
# Block 7: Create the DuckDB views for all IMDb source files.
for view_name, file_name in FILES.items():
   
            create_tsv_view_generic(con, view_name, DATA_DIR / file_name)

print("Views recriadas com sucesso.")

In [ ]:
# Block 8: Recreate the title_principals view explicitly after troubleshooting parser issues.
con.execute("DROP VIEW IF EXISTS title_principals")
create_tsv_view_imdb(con, "title_principals", DATA_DIR / "title.principals.tsv")

In [ ]:
# Block 9: Quick smoke test: read one row from title_principals to confirm that the file is accessible.
try:
    row = con.execute("SELECT * FROM title_principals LIMIT 1").fetchone()
    print("OK - title_principals está acessível")
    print(row)
except Exception as e:
    print("ERRO ao acessar title_principals")
    print(type(e).__name__)
    print(e)

In [ ]:
# Block 10: Count the rows in title_principals to confirm that the full file can be parsed.
try:
    n = con.execute("SELECT COUNT(*) FROM title_principals").fetchone()[0]
    print("OK - contagem concluída")
    print("Número de linhas:", n)
except Exception as e:
    print("ERRO na contagem")
    print(type(e).__name__)
    print(e)

In [ ]:
# Block 11: Define a helper function to count rows safely and capture any parsing error without stopping the notebook.
def safe_count(con, view_name):
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
        print(f"OK - {view_name}: {n}")
        return (view_name, n, "OK")
    except Exception as e:
        print(f"ERRO - {view_name}")
        print(type(e).__name__, e)
        return (view_name, None, f"ERRO: {type(e).__name__}")

In [ ]:
# Block 12: Run the safe row-count check for all source views.
results = []

for view_name in FILES.keys():
    results.append(safe_count(con, view_name))

In [ ]:
# Block 13: Summarize the row-count validation results in a small DataFrame.
import pandas as pd
counts_df = pd.DataFrame(results, columns=["table", "rows", "status"])
counts_df

In [ ]:
# Block 14: Aggregate title.principals at the title level to compute cast/crew cardinality features.
con.execute("""
CREATE OR REPLACE TABLE principals_agg AS
SELECT
    tconst,
    COUNT(*) AS principals_count,
    COUNT(DISTINCT nconst) AS principals_people_count,
    COUNT(DISTINCT category) AS principal_category_count
FROM title_principals
GROUP BY tconst
""")

In [ ]:
# Block 15: Sanity check: confirm that principals_agg was created successfully.
con.execute("SELECT COUNT(*) FROM principals_agg").fetchone()

In [ ]:
# Block 16: Aggregate title.akas at the title level to measure the number of alternative titles, regions, and languages.
con.execute("""
CREATE OR REPLACE TABLE akas_agg AS
SELECT
    titleId AS tconst,
    COUNT(*) AS akas_count,
    COUNT(DISTINCT region) AS akas_region_count,
    COUNT(DISTINCT language) AS akas_language_count
FROM title_akas
GROUP BY titleId
""")

In [ ]:
# Block 17: Sanity check: confirm that akas_agg was created successfully.
con.execute("SELECT COUNT(*) FROM akas_agg").fetchone()

In [ ]:
# Block 18: Aggregate title.crew at the title level by counting listed directors and writers.
con.execute("""
CREATE OR REPLACE TABLE crew_agg AS
SELECT
    tconst,
    CASE
        WHEN directors IS NULL OR directors = '' THEN 0
        ELSE array_length(string_split(directors, ','))
    END AS directors_count,
    CASE
        WHEN writers IS NULL OR writers = '' THEN 0
        ELSE array_length(string_split(writers, ','))
    END AS writers_count
FROM title_crew
""")

In [ ]:
# Block 19: Sanity check: confirm that crew_agg was created successfully.
con.execute("SELECT COUNT(*) FROM crew_agg").fetchone()

In [ ]:
# Block 20: Prepare the ratings table with typed numeric columns and a has_rating flag.
con.execute("""
CREATE OR REPLACE TABLE ratings_agg AS
SELECT
    tconst,
    TRY_CAST(averageRating AS DOUBLE) AS averageRating,
    TRY_CAST(numVotes AS BIGINT) AS numVotes,
    1 AS has_rating
FROM title_ratings
""")

In [ ]:
# Block 21: Sanity check: confirm that ratings_agg was created successfully.
con.execute("SELECT COUNT(*) FROM ratings_agg").fetchone()

In [ ]:
# Block 22: Prepare episode-level information, including parent series and episode numbering.
con.execute("""
CREATE OR REPLACE TABLE episode_agg AS
SELECT
    tconst,
    parentTconst,
    TRY_CAST(seasonNumber AS INTEGER) AS seasonNumber,
    TRY_CAST(episodeNumber AS INTEGER) AS episodeNumber,
    1 AS is_episode
FROM title_episode
""")

In [ ]:
# Block 23: Sanity check: confirm that episode_agg was created successfully.
con.execute("SELECT COUNT(*) FROM episode_agg").fetchone()

In [ ]:
# Block 24: Build one row per title by joining the title basics table with aggregated principals, AKAs, crew, ratings, and episode information.
con.execute("""
CREATE OR REPLACE TABLE title_features AS
SELECT
    b.tconst,
    b.titleType,
    b.primaryTitle,
    b.originalTitle,
    TRY_CAST(b.isAdult AS INTEGER) AS isAdult,
    TRY_CAST(b.startYear AS INTEGER) AS startYear,
    TRY_CAST(b.endYear AS INTEGER) AS endYear,
    TRY_CAST(b.runtimeMinutes AS INTEGER) AS runtimeMinutes,
    b.genres,

    CASE
        WHEN b.genres IS NULL OR b.genres = '' THEN 'unknown'
        ELSE split_part(b.genres, ',', 1)
    END AS primaryGenre,

    CASE
        WHEN TRY_CAST(b.startYear AS INTEGER) IS NULL THEN 'unknown'
        ELSE CAST(CAST(FLOOR(TRY_CAST(b.startYear AS DOUBLE) / 10) * 10 AS BIGINT) AS VARCHAR) || 's'
    END AS decade,

    COALESCE(e.is_episode, 0) AS is_episode,
    e.parentTconst,
    e.seasonNumber,
    e.episodeNumber,

    COALESCE(p.principals_count, 0) AS principals_count,
    COALESCE(p.principals_people_count, 0) AS principals_people_count,
    COALESCE(p.principal_category_count, 0) AS principal_category_count,

    COALESCE(a.akas_count, 0) AS akas_count,
    COALESCE(a.akas_region_count, 0) AS akas_region_count,
    COALESCE(a.akas_language_count, 0) AS akas_language_count,

    COALESCE(c.directors_count, 0) AS directors_count,
    COALESCE(c.writers_count, 0) AS writers_count,

    COALESCE(r.has_rating, 0) AS has_rating,
    r.averageRating,
    r.numVotes

FROM title_basics b
LEFT JOIN principals_agg p USING (tconst)
LEFT JOIN akas_agg a USING (tconst)
LEFT JOIN crew_agg c USING (tconst)
LEFT JOIN episode_agg e USING (tconst)
LEFT JOIN ratings_agg r USING (tconst)
""")

In [ ]:
# Block 25: Sanity check: confirm the number of titles in the integrated title_features table.
con.execute("SELECT COUNT(*) FROM title_features").fetchone()

In [ ]:
# Block 26: Inspect a small sample of title_features to verify that the integrated attributes look correct.
con.execute("SELECT * FROM title_features LIMIT 5").df()

In [ ]:
# Block 27: Inspect the distribution of title types in the integrated title_features table.
con.execute("""
SELECT titleType, COUNT(*) AS n
FROM title_features
GROUP BY titleType
ORDER BY n DESC
LIMIT 20
""").df()

In [ ]:
# Block 28: Inspect the distribution of primary genres in the integrated title_features table.
con.execute("""
SELECT primaryGenre, COUNT(*) AS n
FROM title_features
GROUP BY primaryGenre
ORDER BY n DESC
LIMIT 20
""").df()

In [ ]:
# Block 29: Check how many titles are episodes versus non-episodes.
con.execute("""
SELECT is_episode, COUNT(*) AS n
FROM title_features
GROUP BY is_episode
ORDER BY is_episode
""").df()

In [ ]:
# Block 30: Check how many titles have a rating record.
con.execute("""
SELECT has_rating, COUNT(*) AS n
FROM title_features
GROUP BY has_rating
ORDER BY has_rating
""").df()

In [ ]:
# Block 31: Create the sampling base by deriving coarse buckets for principals count, vote count, and runtime.
con.execute("""
CREATE OR REPLACE TABLE title_sampling_base AS
SELECT
    *,
    
    CASE
        WHEN principals_count = 0 THEN '0'
        WHEN principals_count = 1 THEN '1'
        WHEN principals_count BETWEEN 2 AND 3 THEN '2_3'
        WHEN principals_count BETWEEN 4 AND 6 THEN '4_6'
        WHEN principals_count BETWEEN 7 AND 10 THEN '7_10'
        ELSE '11_plus'
    END AS principals_bucket,

    CASE
        WHEN numVotes IS NULL OR numVotes = 0 THEN '0'
        WHEN numVotes BETWEEN 1 AND 9 THEN '1_9'
        WHEN numVotes BETWEEN 10 AND 99 THEN '10_99'
        WHEN numVotes BETWEEN 100 AND 999 THEN '100_999'
        WHEN numVotes BETWEEN 1000 AND 9999 THEN '1000_9999'
        ELSE '10000_plus'
    END AS numVotes_bucket,

    CASE
        WHEN runtimeMinutes IS NULL THEN 'unknown'
        WHEN runtimeMinutes < 10 THEN 'lt_10'
        WHEN runtimeMinutes < 30 THEN '10_29'
        WHEN runtimeMinutes < 60 THEN '30_59'
        WHEN runtimeMinutes < 120 THEN '60_119'
        ELSE '120_plus'
    END AS runtime_bucket

FROM title_features
""")

In [ ]:
# Block 32: Sanity check: confirm that title_sampling_base was created successfully.
con.execute("SELECT COUNT(*) FROM title_sampling_base").fetchone()

In [ ]:
# Block 33: Inspect the distribution of the principals-count buckets used for stratification.
con.execute("""
SELECT principals_bucket, COUNT(*) AS n
FROM title_sampling_base
GROUP BY principals_bucket
ORDER BY n DESC
""").df()

In [ ]:
# Block 34: Inspect the distribution of the numVotes buckets used for stratification.
con.execute("""
SELECT numVotes_bucket, COUNT(*) AS n
FROM title_sampling_base
GROUP BY numVotes_bucket
ORDER BY n DESC
""").df()

In [ ]:
# Block 35: Inspect the distribution of the runtime buckets used for stratification.
con.execute("""
SELECT runtime_bucket, COUNT(*) AS n
FROM title_sampling_base
GROUP BY runtime_bucket
ORDER BY n DESC
""").df()

In [ ]:
# Block 36: Create the final stratification key by combining title type, decade, primary genre, rating availability, and principals bucket.
con.execute("""
CREATE OR REPLACE TABLE title_sampling_strata AS
SELECT
    *,
    CONCAT_WS(
        '|',
        COALESCE(titleType, 'unknown'),
        COALESCE(decade, 'unknown'),
        COALESCE(primaryGenre, 'unknown'),
        CAST(COALESCE(has_rating, 0) AS VARCHAR),
        COALESCE(principals_bucket, 'unknown')
    ) AS stratum_key
FROM title_sampling_base
""")

In [ ]:
# Block 37: Execute this block as part of the IMDb scale-factor generation and validation pipeline.
con.execute("SELECT COUNT(*) FROM title_sampling_strata").fetchone()

In [ ]:
# Block 38: Check how many distinct strata were produced by the stratification design.
con.execute("""
SELECT COUNT(DISTINCT stratum_key) AS n_strata
FROM title_sampling_strata
""").df()

In [ ]:
# Block 39: Inspect the largest strata to understand where most titles are concentrated.
con.execute("""
SELECT stratum_key, COUNT(*) AS n
FROM title_sampling_strata
GROUP BY stratum_key
ORDER BY n DESC
LIMIT 20
""").df()

In [ ]:
# Block 40: Define a fixed random seed so the sampling process is reproducible.
SEED = "imdb_sf_seed_v1"

In [ ]:
# Block 41: Assign a deterministic random order to titles within each stratum and compute the stratum size.
con.execute(f"""
CREATE OR REPLACE TABLE title_sampling_ranked AS
SELECT
    *,
    ROW_NUMBER() OVER (
        PARTITION BY stratum_key
        ORDER BY hash(tconst || '{SEED}')
    ) AS rn_in_stratum,
    COUNT(*) OVER (
        PARTITION BY stratum_key
    ) AS n_in_stratum
FROM title_sampling_strata
""")

In [ ]:
# Block 42: Sanity check: confirm that the ranked sampling table was created successfully.
con.execute("SELECT COUNT(*) FROM title_sampling_ranked").fetchone()

In [ ]:
# Block 43: Flag which titles belong to SF0.25, SF0.5, and SF1 within each stratum.
con.execute("""
CREATE OR REPLACE TABLE title_sampling_flags AS
SELECT
    *,
    
    CASE
        WHEN rn_in_stratum <= GREATEST(1, CAST(ROUND(n_in_stratum * 0.25) AS BIGINT))
        THEN 1 ELSE 0
    END AS in_sf_025,

    CASE
        WHEN rn_in_stratum <= GREATEST(1, CAST(ROUND(n_in_stratum * 0.50) AS BIGINT))
        THEN 1 ELSE 0
    END AS in_sf_05,

    1 AS in_sf_1

FROM title_sampling_ranked
""")

In [ ]:
# Block 44: Sanity check: confirm that the sampling flags table was created successfully.
con.execute("SELECT COUNT(*) FROM title_sampling_flags").fetchone()

In [ ]:
# Block 45: Check the approximate number of titles selected for each scale factor before referential closure.
con.execute("""
SELECT
    SUM(in_sf_025) AS sf_025_titles,
    SUM(in_sf_05)  AS sf_05_titles,
    SUM(in_sf_1)   AS sf_1_titles,
    COUNT(*)       AS total_titles
FROM title_sampling_flags
""").df()

In [ ]:
# Block 46: Validate that the sampled subsets are nested: SF0.25 must be a subset of SF0.5, and SF0.5 must be a subset of SF1.
con.execute("""
SELECT
    SUM(CASE WHEN in_sf_025 = 1 AND in_sf_05 = 0 THEN 1 ELSE 0 END) AS bad_025_not_in_05,
    SUM(CASE WHEN in_sf_05 = 1 AND in_sf_1 = 0 THEN 1 ELSE 0 END) AS bad_05_not_in_1
FROM title_sampling_flags
""").df()

In [ ]:
# Block 47: Create the initial title list for SF0.25 directly from the sampling flags.
con.execute("""
CREATE OR REPLACE TABLE sf_025_titles_initial AS
SELECT tconst
FROM title_sampling_flags
WHERE in_sf_025 = 1
""")

In [ ]:
# Block 48: Create the initial title list for SF0.5 directly from the sampling flags.
con.execute("""
CREATE OR REPLACE TABLE sf_05_titles_initial AS
SELECT tconst
FROM title_sampling_flags
WHERE in_sf_05 = 1
""")

In [ ]:
# Block 49: Create the initial title list for SF1 directly from the sampling flags.
con.execute("""
CREATE OR REPLACE TABLE sf_1_titles_initial AS
SELECT tconst
FROM title_sampling_flags
WHERE in_sf_1 = 1
""")

In [ ]:
# Block 50: Create the initial title list for SF0.25 directly from the sampling flags.
con.execute("""
CREATE OR REPLACE TABLE sf_025_titles AS
SELECT DISTINCT tconst
FROM (
    SELECT tconst
    FROM sf_025_titles_initial

    UNION

    SELECT parentTconst AS tconst
    FROM title_features
    WHERE is_episode = 1
      AND parentTconst IS NOT NULL
      AND tconst IN (SELECT tconst FROM sf_025_titles_initial)
)
WHERE tconst IS NOT NULL
""")

In [ ]:
# Block 51: Create the initial title list for SF0.5 directly from the sampling flags.
con.execute("""
CREATE OR REPLACE TABLE sf_05_titles AS
SELECT DISTINCT tconst
FROM (
    SELECT tconst
    FROM sf_05_titles_initial

    UNION

    SELECT parentTconst AS tconst
    FROM title_features
    WHERE is_episode = 1
      AND parentTconst IS NOT NULL
      AND tconst IN (SELECT tconst FROM sf_05_titles_initial)
)
WHERE tconst IS NOT NULL
""")

In [ ]:
# Block 52: Create the initial title list for SF1 directly from the sampling flags.
con.execute("""
CREATE OR REPLACE TABLE sf_1_titles AS
SELECT DISTINCT tconst
FROM sf_1_titles_initial
WHERE tconst IS NOT NULL
""")

In [ ]:
# Block 53: Compare the number of titles across the three scale factors after referential closure.
con.execute("""
SELECT 'SF0.25' AS sf, COUNT(*) AS n FROM sf_025_titles
UNION ALL
SELECT 'SF0.5'  AS sf, COUNT(*) AS n FROM sf_05_titles
UNION ALL
SELECT 'SF1'    AS sf, COUNT(*) AS n FROM sf_1_titles
""").df()

In [ ]:
# Block 54: Inspect the distribution of title types in the integrated title_features table.
con.execute("""
WITH full_dist AS (
    SELECT titleType, COUNT(*) AS n_full
    FROM title_features
    GROUP BY titleType
),
sf025_dist AS (
    SELECT f.titleType, COUNT(*) AS n_025
    FROM title_features f
    JOIN sf_025_titles s USING (tconst)
    GROUP BY f.titleType
),
sf05_dist AS (
    SELECT f.titleType, COUNT(*) AS n_05
    FROM title_features f
    JOIN sf_05_titles s USING (tconst)
    GROUP BY f.titleType
)
SELECT
    COALESCE(full_dist.titleType, sf025_dist.titleType, sf05_dist.titleType) AS titleType,
    n_full,
    n_025,
    n_05
FROM full_dist
FULL OUTER JOIN sf025_dist USING (titleType)
FULL OUTER JOIN sf05_dist USING (titleType)
ORDER BY n_full DESC
""").df()

In [ ]:
# Block 55: Inspect the distribution of primary genres in the integrated title_features table.
con.execute("""
WITH full_dist AS (
    SELECT primaryGenre, COUNT(*) AS n_full
    FROM title_features
    GROUP BY primaryGenre
),
sf025_dist AS (
    SELECT f.primaryGenre, COUNT(*) AS n_025
    FROM title_features f
    JOIN sf_025_titles s USING (tconst)
    GROUP BY f.primaryGenre
),
sf05_dist AS (
    SELECT f.primaryGenre, COUNT(*) AS n_05
    FROM title_features f
    JOIN sf_05_titles s USING (tconst)
    GROUP BY f.primaryGenre
)
SELECT
    COALESCE(full_dist.primaryGenre, sf025_dist.primaryGenre, sf05_dist.primaryGenre) AS primaryGenre,
    n_full,
    n_025,
    n_05
FROM full_dist
FULL OUTER JOIN sf025_dist USING (primaryGenre)
FULL OUTER JOIN sf05_dist USING (primaryGenre)
ORDER BY n_full DESC
LIMIT 20
""").df()

In [ ]:
# Block 56: Execute this block as part of the IMDb scale-factor generation and validation pipeline.
con.execute("""
SELECT
    (SELECT COUNT(*) FROM sf_025_titles WHERE tconst NOT IN (SELECT tconst FROM sf_05_titles)) AS sf025_not_in_sf05,
    (SELECT COUNT(*) FROM sf_05_titles WHERE tconst NOT IN (SELECT tconst FROM sf_1_titles))   AS sf05_not_in_sf1
""").df()

In [ ]:
# Block 57: Execute this block as part of the IMDb scale-factor generation and validation pipeline.
con.execute("""
SELECT tconst
FROM sf_05_titles
WHERE tconst NOT IN (SELECT tconst FROM sf_1_titles)
""").df()

In [ ]:
# Block 58: Execute this block as part of the IMDb scale-factor generation and validation pipeline.
con.execute("""
SELECT *
FROM title_features
WHERE tconst IN (
    SELECT tconst
    FROM sf_05_titles
    WHERE tconst NOT IN (SELECT tconst FROM sf_1_titles)
)
""").df()

In [ ]:
# Block 59: Collect the people referenced by title.principals in SF0.25.
con.execute("""
CREATE OR REPLACE TABLE sf_025_people_from_principals AS
SELECT DISTINCT p.nconst
FROM title_principals p
JOIN sf_025_titles t USING (tconst)
WHERE p.nconst IS NOT NULL
""")

In [ ]:
# Block 60: Collect the people referenced by title.principals in SF0.5.
con.execute("""
CREATE OR REPLACE TABLE sf_05_people_from_principals AS
SELECT DISTINCT p.nconst
FROM title_principals p
JOIN sf_05_titles t USING (tconst)
WHERE p.nconst IS NOT NULL
""")

In [ ]:
# Block 61: Collect the people referenced by title.principals in SF1.
con.execute("""
CREATE OR REPLACE TABLE sf_1_people_from_principals AS
SELECT DISTINCT p.nconst
FROM title_principals p
JOIN sf_1_titles t USING (tconst)
WHERE p.nconst IS NOT NULL
""")

In [ ]:
# Block 62: Collect directors and writers referenced in title.crew for SF0.25.
con.execute("""
CREATE OR REPLACE TABLE sf_025_people_from_crew AS
WITH crew_rows AS (
    SELECT c.tconst, c.directors, c.writers
    FROM title_crew c
    JOIN sf_025_titles t USING (tconst)
),
director_people AS (
    SELECT trim(x) AS nconst
    FROM crew_rows, UNNEST(string_split(COALESCE(directors, ''), ',')) AS u(x)
),
writer_people AS (
    SELECT trim(x) AS nconst
    FROM crew_rows, UNNEST(string_split(COALESCE(writers, ''), ',')) AS u(x)
)
SELECT DISTINCT nconst
FROM (
    SELECT nconst FROM director_people
    UNION ALL
    SELECT nconst FROM writer_people
)
WHERE nconst IS NOT NULL
  AND nconst <> ''
""")

In [ ]:
# Block 63: Collect directors and writers referenced in title.crew for SF0.5.
con.execute("""
CREATE OR REPLACE TABLE sf_05_people_from_crew AS
WITH crew_rows AS (
    SELECT c.tconst, c.directors, c.writers
    FROM title_crew c
    JOIN sf_05_titles t USING (tconst)
),
director_people AS (
    SELECT trim(x) AS nconst
    FROM crew_rows, UNNEST(string_split(COALESCE(directors, ''), ',')) AS u(x)
),
writer_people AS (
    SELECT trim(x) AS nconst
    FROM crew_rows, UNNEST(string_split(COALESCE(writers, ''), ',')) AS u(x)
)
SELECT DISTINCT nconst
FROM (
    SELECT nconst FROM director_people
    UNION ALL
    SELECT nconst FROM writer_people
)
WHERE nconst IS NOT NULL
  AND nconst <> ''
""")

In [ ]:
# Block 64: Collect directors and writers referenced in title.crew for SF1.
con.execute("""
CREATE OR REPLACE TABLE sf_1_people_from_crew AS
WITH crew_rows AS (
    SELECT c.tconst, c.directors, c.writers
    FROM title_crew c
    JOIN sf_1_titles t USING (tconst)
),
director_people AS (
    SELECT trim(x) AS nconst
    FROM crew_rows, UNNEST(string_split(COALESCE(directors, ''), ',')) AS u(x)
),
writer_people AS (
    SELECT trim(x) AS nconst
    FROM crew_rows, UNNEST(string_split(COALESCE(writers, ''), ',')) AS u(x)
)
SELECT DISTINCT nconst
FROM (
    SELECT nconst FROM director_people
    UNION ALL
    SELECT nconst FROM writer_people
)
WHERE nconst IS NOT NULL
  AND nconst <> ''
""")

In [ ]:
# Block 65: Create the final set of people required for SF0.25 by merging principals- and crew-based references.
con.execute("""
CREATE OR REPLACE TABLE sf_025_people AS
SELECT DISTINCT nconst
FROM (
    SELECT nconst FROM sf_025_people_from_principals
    UNION
    SELECT nconst FROM sf_025_people_from_crew
)
WHERE nconst IS NOT NULL
""")

In [ ]:
# Block 66: Create the final set of people required for SF0.5 by merging principals- and crew-based references.
con.execute("""
CREATE OR REPLACE TABLE sf_05_people AS
SELECT DISTINCT nconst
FROM (
    SELECT nconst FROM sf_05_people_from_principals
    UNION
    SELECT nconst FROM sf_05_people_from_crew
)
WHERE nconst IS NOT NULL
""")

In [ ]:
# Block 67: Create the final set of people required for SF1 by merging principals- and crew-based references.
con.execute("""
CREATE OR REPLACE TABLE sf_1_people AS
SELECT DISTINCT nconst
FROM (
    SELECT nconst FROM sf_1_people_from_principals
    UNION
    SELECT nconst FROM sf_1_people_from_crew
)
WHERE nconst IS NOT NULL
""")

In [ ]:
# Block 68: Compare the number of referenced people across the three scale factors.
con.execute("""
SELECT 'SF0.25' AS sf, COUNT(*) AS n FROM sf_025_people
UNION ALL
SELECT 'SF0.5', COUNT(*) FROM sf_05_people
UNION ALL
SELECT 'SF1', COUNT(*) FROM sf_1_people
""").df()

In [ ]:
# Block 69: Create the output directories where the exported TSV files for each scale factor will be stored.
from pathlib import Path

OUT_DIR = DATA_DIR / "sf_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_025 = OUT_DIR / "sf_025"
OUT_05  = OUT_DIR / "sf_05"
OUT_1   = OUT_DIR / "sf_1"

OUT_025.mkdir(parents=True, exist_ok=True)
OUT_05.mkdir(parents=True, exist_ok=True)
OUT_1.mkdir(parents=True, exist_ok=True)

In [ ]:
# Block 70: Define a helper function to export any SQL query result to a TSV file.
def export_query_to_tsv(con, query, out_path):
    out_path = Path(out_path)
    safe_out = str(out_path).replace("\\", "/").replace("'", "''")
    sql = f"""
    COPY (
        {query}
    ) TO '{safe_out}'
    WITH (
        FORMAT CSV,
        DELIMITER '\t',
        HEADER TRUE,
        NULL '\\N'
    )
    """
    con.execute(sql)

In [ ]:
# Block 71: Export name.basics.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM name_basics
WHERE nconst IN (SELECT nconst FROM sf_025_people)
""", OUT_025 / "name.basics.tsv")

In [ ]:
# Block 72: Export title.akas.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM title_akas
WHERE titleId IN (SELECT tconst FROM sf_025_titles)
""", OUT_025 / "title.akas.tsv")

In [ ]:
# Block 73: Export title.basics.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM title_basics
WHERE tconst IN (SELECT tconst FROM sf_025_titles)
""", OUT_025 / "title.basics.tsv")

In [ ]:
# Block 74: Export title.crew.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM title_crew
WHERE tconst IN (SELECT tconst FROM sf_025_titles)
""", OUT_025 / "title.crew.tsv")

In [ ]:
# Block 75: Export title.episode.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM title_episode
WHERE tconst IN (SELECT tconst FROM sf_025_titles)
""", OUT_025 / "title.episode.tsv")

In [ ]:
# Block 76: Export title.principals.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM title_principals
WHERE tconst IN (SELECT tconst FROM sf_025_titles)
""", OUT_025 / "title.principals.tsv")

In [ ]:
# Block 77: Export title.ratings.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM title_ratings
WHERE tconst IN (SELECT tconst FROM sf_025_titles)
""", OUT_025 / "title.ratings.tsv")

In [ ]:
# Block 78: Export name.basics.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM name_basics
WHERE nconst IN (SELECT nconst FROM sf_05_people)
""", OUT_05 / "name.basics.tsv")

In [ ]:
# Block 79: Export title.akas.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM title_akas
WHERE titleId IN (SELECT tconst FROM sf_05_titles)
""", OUT_05 / "title.akas.tsv")

In [ ]:
# Block 80: Export title.basics.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM title_basics
WHERE tconst IN (SELECT tconst FROM sf_05_titles)
""", OUT_05 / "title.basics.tsv")

In [ ]:
# Block 81: Export title.crew.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM title_crew
WHERE tconst IN (SELECT tconst FROM sf_05_titles)
""", OUT_05 / "title.crew.tsv")

In [ ]:
# Block 82: Export title.episode.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM title_episode
WHERE tconst IN (SELECT tconst FROM sf_05_titles)
""", OUT_05 / "title.episode.tsv")

In [ ]:
# Block 83: Export title.principals.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM title_principals
WHERE tconst IN (SELECT tconst FROM sf_05_titles)
""", OUT_05 / "title.principals.tsv")

In [ ]:
# Block 84: Export title.ratings.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM title_ratings
WHERE tconst IN (SELECT tconst FROM sf_05_titles)
""", OUT_05 / "title.ratings.tsv")

In [ ]:
# Block 85: Export name.basics.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM name_basics
WHERE nconst IN (SELECT nconst FROM sf_1_people)
""", OUT_1 / "name.basics.tsv")

In [ ]:
# Block 86: Export title.akas.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM title_akas
WHERE titleId IN (SELECT tconst FROM sf_1_titles)
""", OUT_1 / "title.akas.tsv")

In [ ]:
# Block 87: Export title.basics.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM title_basics
WHERE tconst IN (SELECT tconst FROM sf_1_titles)
""", OUT_1 / "title.basics.tsv")

In [ ]:
# Block 88: Export title.crew.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM title_crew
WHERE tconst IN (SELECT tconst FROM sf_1_titles)
""", OUT_1 / "title.crew.tsv")

In [ ]:
# Block 89: Export title.episode.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM title_episode
WHERE tconst IN (SELECT tconst FROM sf_1_titles)
""", OUT_1 / "title.episode.tsv")

In [ ]:
# Block 90: Export title.principals.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM title_principals
WHERE tconst IN (SELECT tconst FROM sf_1_titles)
""", OUT_1 / "title.principals.tsv")

In [ ]:
# Block 91: Export title.ratings.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM title_ratings
WHERE tconst IN (SELECT tconst FROM sf_1_titles)
""", OUT_1 / "title.ratings.tsv")

In [ ]:
# Block 92: List all exported TSV files to verify that the export step created the expected artifacts.
for p in sorted(OUT_DIR.rglob("*.tsv")):
    print(p)

In [ ]:
# Block 93: List all exported TSV files to verify that the export step created the expected artifacts.
size_rows = []
for p in sorted(OUT_DIR.rglob("*.tsv")):
    size_rows.append((str(p), p.stat().st_size))

pd.DataFrame(size_rows, columns=["file", "bytes"]).sort_values("file")

In [ ]:
# Block 94: Define the expected export structure and register the folders for the three scale factors.
from pathlib import Path
import pandas as pd

expected_files = [
    "name.basics.tsv",
    "title.akas.tsv",
    "title.basics.tsv",
    "title.crew.tsv",
    "title.episode.tsv",
    "title.principals.tsv",
    "title.ratings.tsv",
]

sf_dirs = {
    "SF0.25": OUT_025,
    "SF0.5": OUT_05,
    "SF1": OUT_1,
}

rows = []
for sf, folder in sf_dirs.items():
    for fname in expected_files:
        p = folder / fname
        rows.append({
            "sf": sf,
            "file": fname,
            "exists": p.exists(),
            "size_bytes": p.stat().st_size if p.exists() else None
        })

files_check_df = pd.DataFrame(rows)
files_check_df

In [ ]:
# Block 95: Count the number of data rows in each exported TSV file and summarize the result.
def count_tsv_rows(path):
    with open(path, "r", encoding="utf-8", newline="") as f:
        # Subtract the header row
        return sum(1 for _ in f) - 1

count_rows = []
for sf, folder in sf_dirs.items():
    for fname in expected_files:
        p = folder / fname
        if p.exists():
            n = count_tsv_rows(p)
        else:
            n = None
        count_rows.append({
            "sf": sf,
            "file": fname,
            "rows": n
        })

export_counts_df = pd.DataFrame(count_rows)
export_counts_df

In [ ]:
# Block 96: Check monotonicity: every exported file should have non-decreasing row counts from SF0.25 to SF0.5 to SF1.
pivot_counts = export_counts_df.pivot(index="file", columns="sf", values="rows").reset_index()
pivot_counts["ok_monotonic"] = (
    (pivot_counts["SF0.25"] <= pivot_counts["SF0.5"]) &
    (pivot_counts["SF0.5"] <= pivot_counts["SF1"])
)
pivot_counts

In [ ]:
# Block 97: Register the exported TSV files back into DuckDB as views so they can be validated.
def register_export_views(con, folder, prefix):
    folder = Path(folder)

    create_tsv_view_imdb(con, f"{prefix}_name_basics", folder / "name.basics.tsv")
    create_tsv_view_imdb(con, f"{prefix}_title_akas", folder / "title.akas.tsv")
    create_tsv_view_imdb(con, f"{prefix}_title_basics", folder / "title.basics.tsv")
    create_tsv_view_imdb(con, f"{prefix}_title_crew", folder / "title.crew.tsv")
    create_tsv_view_imdb(con, f"{prefix}_title_episode", folder / "title.episode.tsv")
    create_tsv_view_imdb(con, f"{prefix}_title_principals", folder / "title.principals.tsv")
    create_tsv_view_imdb(con, f"{prefix}_title_ratings", folder / "title.ratings.tsv")

In [ ]:
# Block 98: Register the exported files for SF0.25, SF0.5, and SF1.
register_export_views(con, OUT_025, "sf025")
register_export_views(con, OUT_05, "sf05")
register_export_views(con, OUT_1, "sf1")

In [ ]:
# Block 99: Define a detailed referential-consistency validation using LEFT JOIN checks between the exported files.
def validate_scale_left_join(con, prefix):
    checks = []

    q1 = f"""
    SELECT COUNT(*)
    FROM {prefix}_title_akas a
    LEFT JOIN {prefix}_title_basics b
      ON a.titleId = b.tconst
    WHERE a.titleId IS NOT NULL
      AND b.tconst IS NULL
    """
    checks.append(("akas_title_missing", con.execute(q1).fetchone()[0]))

    q2 = f"""
    SELECT COUNT(*)
    FROM {prefix}_title_principals p
    LEFT JOIN {prefix}_title_basics b
      ON p.tconst = b.tconst
    WHERE p.tconst IS NOT NULL
      AND b.tconst IS NULL
    """
    checks.append(("principals_title_missing", con.execute(q2).fetchone()[0]))

    q3 = f"""
    SELECT COUNT(*)
    FROM {prefix}_title_principals p
    LEFT JOIN {prefix}_name_basics n
      ON p.nconst = n.nconst
    WHERE p.nconst IS NOT NULL
      AND n.nconst IS NULL
    """
    checks.append(("principals_name_missing", con.execute(q3).fetchone()[0]))

    q4 = f"""
    SELECT COUNT(*)
    FROM {prefix}_title_crew c
    LEFT JOIN {prefix}_title_basics b
      ON c.tconst = b.tconst
    WHERE c.tconst IS NOT NULL
      AND b.tconst IS NULL
    """
    checks.append(("crew_title_missing", con.execute(q4).fetchone()[0]))

    q5 = f"""
    SELECT COUNT(*)
    FROM {prefix}_title_ratings r
    LEFT JOIN {prefix}_title_basics b
      ON r.tconst = b.tconst
    WHERE r.tconst IS NOT NULL
      AND b.tconst IS NULL
    """
    checks.append(("ratings_title_missing", con.execute(q5).fetchone()[0]))

    q6 = f"""
    SELECT COUNT(*)
    FROM {prefix}_title_episode e
    LEFT JOIN {prefix}_title_basics b
      ON e.tconst = b.tconst
    WHERE e.tconst IS NOT NULL
      AND b.tconst IS NULL
    """
    checks.append(("episode_title_missing", con.execute(q6).fetchone()[0]))

    q7 = f"""
    SELECT COUNT(*)
    FROM {prefix}_title_episode e
    LEFT JOIN {prefix}_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
    """
    checks.append(("episode_parent_missing", con.execute(q7).fetchone()[0]))

    return pd.DataFrame(checks, columns=["check", "missing_rows"])

In [ ]:
# Block 100: Run the detailed exported-data validation for SF0.25.
val_025 = validate_scale(con, "sf025")

print("SF0.25")
display(val_025)



In [ ]:
# Block 101: Run the detailed exported-data validation for SF0.5.
val_05 = validate_scale(con, "sf05")

print("SF0.5")
display(val_05)



In [ ]:
# Block 102: Run the detailed exported-data validation for SF1.
val_1 = validate_scale(con, "sf1")
print("SF1")
display(val_1)

In [ ]:
# Block 103: Check whether missing person references already exist in the original IMDb snapshot.
q = """
SELECT COUNT(*)
FROM title_principals p
LEFT JOIN name_basics n
  ON p.nconst = n.nconst
WHERE p.nconst IS NOT NULL
  AND n.nconst IS NULL
"""
n = con.execute(q).fetchone()[0]
print("principals_name_missing_original =", n)

In [ ]:
# Block 104: Check whether orphan episode-parent references already exist in the original IMDb snapshot.
q = """
SELECT COUNT(*)
FROM title_episode e
LEFT JOIN title_basics b
  ON e.parentTconst = b.tconst
WHERE e.parentTconst IS NOT NULL
  AND b.tconst IS NULL
"""
n = con.execute(q).fetchone()[0]
print("episode_parent_missing_original =", n)

In [ ]:
# Block 105: Build and save a summary table describing the existence, size, and row counts of the exported files.
summary_rows = []

for sf, folder in sf_dirs.items():
    for fname in expected_files:
        p = folder / fname
        summary_rows.append({
            "sf": sf,
            "file": fname,
            "exists": p.exists(),
            "size_bytes": p.stat().st_size if p.exists() else None,
            "rows": count_tsv_rows(p) if p.exists() else None
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "export_summary.csv", index=False)
summary_df

In [ ]:
# Block 106: Export title.episode.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT e.*
FROM title_episode e
JOIN sf_025_titles t
  ON e.tconst = t.tconst
JOIN sf_025_titles p
  ON e.parentTconst = p.tconst
""", OUT_025 / "title.episode.tsv")

In [ ]:
# Block 107: Export title.episode.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT e.*
FROM title_episode e
JOIN sf_05_titles t
  ON e.tconst = t.tconst
JOIN sf_05_titles p
  ON e.parentTconst = p.tconst
""", OUT_05 / "title.episode.tsv")

In [ ]:
# Block 108: Export title.episode.tsv for SF1.
export_query_to_tsv(con, """
SELECT e.*
FROM title_episode e
JOIN sf_1_titles t
  ON e.tconst = t.tconst
JOIN sf_1_titles p
  ON e.parentTconst = p.tconst
""", OUT_1 / "title.episode.tsv")

In [ ]:
# Block 109: Export name.basics.tsv for SF0.25.
export_query_to_tsv(con, """
WITH base_people AS (
    SELECT
        nconst,
        primaryName,
        birthYear,
        deathYear,
        primaryProfession,
        knownForTitles
    FROM name_basics
    WHERE nconst IN (SELECT nconst FROM sf_025_people)
),
missing_people AS (
    SELECT p.nconst
    FROM sf_025_people p
    LEFT JOIN name_basics n
      ON p.nconst = n.nconst
    WHERE p.nconst IS NOT NULL
      AND n.nconst IS NULL
),
stub_people AS (
    SELECT
        nconst,
        NULL AS primaryName,
        NULL AS birthYear,
        NULL AS deathYear,
        NULL AS primaryProfession,
        NULL AS knownForTitles
    FROM missing_people
)
SELECT * FROM base_people
UNION ALL
SELECT * FROM stub_people
""", OUT_025 / "name.basics.tsv")

In [ ]:
# Block 110: Export name.basics.tsv for SF0.5.
export_query_to_tsv(con, """
WITH base_people AS (
    SELECT
        nconst,
        primaryName,
        birthYear,
        deathYear,
        primaryProfession,
        knownForTitles
    FROM name_basics
    WHERE nconst IN (SELECT nconst FROM sf_05_people)
),
missing_people AS (
    SELECT p.nconst
    FROM sf_05_people p
    LEFT JOIN name_basics n
      ON p.nconst = n.nconst
    WHERE p.nconst IS NOT NULL
      AND n.nconst IS NULL
),
stub_people AS (
    SELECT
        nconst,
        NULL AS primaryName,
        NULL AS birthYear,
        NULL AS deathYear,
        NULL AS primaryProfession,
        NULL AS knownForTitles
    FROM missing_people
)
SELECT * FROM base_people
UNION ALL
SELECT * FROM stub_people
""", OUT_05 / "name.basics.tsv")

In [ ]:
# Block 111: Export name.basics.tsv for SF1.
export_query_to_tsv(con, """
WITH base_people AS (
    SELECT
        nconst,
        primaryName,
        birthYear,
        deathYear,
        primaryProfession,
        knownForTitles
    FROM name_basics
    WHERE nconst IN (SELECT nconst FROM sf_1_people)
),
missing_people AS (
    SELECT p.nconst
    FROM sf_1_people p
    LEFT JOIN name_basics n
      ON p.nconst = n.nconst
    WHERE p.nconst IS NOT NULL
      AND n.nconst IS NULL
),
stub_people AS (
    SELECT
        nconst,
        NULL AS primaryName,
        NULL AS birthYear,
        NULL AS deathYear,
        NULL AS primaryProfession,
        NULL AS knownForTitles
    FROM missing_people
)
SELECT * FROM base_people
UNION ALL
SELECT * FROM stub_people
""", OUT_1 / "name.basics.tsv")

In [ ]:
# Block 112: Register the exported files for SF0.25, SF0.5, and SF1.
register_export_views(con, OUT_025, "sf025")
register_export_views(con, OUT_05, "sf05")
register_export_views(con, OUT_1, "sf1")

In [ ]:
# Block 113: Define a lightweight final validation focused on the two repaired integrity constraints.
def validate_scale_light(con, prefix):
    checks = {}

    checks["principals_name_missing"] = con.execute(f"""
        SELECT COUNT(*)
        FROM {prefix}_title_principals p
        LEFT JOIN {prefix}_name_basics n
          ON p.nconst = n.nconst
        WHERE p.nconst IS NOT NULL
          AND n.nconst IS NULL
    """).fetchone()[0]

    checks["episode_parent_missing"] = con.execute(f"""
        SELECT COUNT(*)
        FROM {prefix}_title_episode e
        LEFT JOIN {prefix}_title_basics b
          ON e.parentTconst = b.tconst
        WHERE e.parentTconst IS NOT NULL
          AND b.tconst IS NULL
    """).fetchone()[0]

    return checks

In [ ]:
# Block 114: Run the lightweight final validation for the three scale factors.
print("SF0.25", validate_scale_light(con, "sf025"))
print("SF0.5",  validate_scale_light(con, "sf05"))
print("SF1",    validate_scale_light(con, "sf1"))

In [ ]:
# Block 115: Count how many stub person records were added to each scale factor.
print("SF0.25 stubs =", con.execute("""
SELECT COUNT(*)
FROM sf_025_people p
LEFT JOIN name_basics n
  ON p.nconst = n.nconst
WHERE p.nconst IS NOT NULL
  AND n.nconst IS NULL
""").fetchone()[0])

print("SF0.5 stubs =", con.execute("""
SELECT COUNT(*)
FROM sf_05_people p
LEFT JOIN name_basics n
  ON p.nconst = n.nconst
WHERE p.nconst IS NOT NULL
  AND n.nconst IS NULL
""").fetchone()[0])

print("SF1 stubs =", con.execute("""
SELECT COUNT(*)
FROM sf_1_people p
LEFT JOIN name_basics n
  ON p.nconst = n.nconst
WHERE p.nconst IS NOT NULL
  AND n.nconst IS NULL
""").fetchone()[0])

In [ ]:
# Block 116: Export title.basics.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT *
FROM title_basics
WHERE tconst IN (SELECT tconst FROM sf_025_titles)
""", OUT_025 / "title.basics.tsv")

In [ ]:
# Block 117: Export title.basics.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT *
FROM title_basics
WHERE tconst IN (SELECT tconst FROM sf_05_titles)
""", OUT_05 / "title.basics.tsv")

In [ ]:
# Block 118: Export title.basics.tsv for SF1.
export_query_to_tsv(con, """
SELECT *
FROM title_basics
WHERE tconst IN (SELECT tconst FROM sf_1_titles)
""", OUT_1 / "title.basics.tsv")

In [ ]:
# Block 119: Check whether any orphan parent references remain after the repair step.
print("SF0.25 episode_parent_missing =", con.execute("""
    SELECT COUNT(*)
    FROM sf025_title_episode e
    LEFT JOIN sf025_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
""").fetchone()[0])

print("SF0.5 episode_parent_missing =", con.execute("""
    SELECT COUNT(*)
    FROM sf05_title_episode e
    LEFT JOIN sf05_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
""").fetchone()[0])

print("SF1 episode_parent_missing =", con.execute("""
    SELECT COUNT(*)
    FROM sf1_title_episode e
    LEFT JOIN sf1_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
""").fetchone()[0])

In [ ]:
# Block 120: Inspect the remaining problematic parent title identifiers after validation.
print("SF0.25 problem parent =", con.execute("""
    SELECT DISTINCT e.parentTconst
    FROM sf025_title_episode e
    LEFT JOIN sf025_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
    LIMIT 5
""").fetchall())

In [ ]:
# Block 121: Inspect the remaining problematic parent title identifiers after validation.
print("SF0.5 problem parent =", con.execute("""
    SELECT DISTINCT e.parentTconst
    FROM sf05_title_episode e
    LEFT JOIN sf05_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
    LIMIT 5
""").fetchall())

In [ ]:
# Block 122: Export title.episode.tsv for SF0.25.
export_query_to_tsv(con, """
SELECT e.*
FROM title_episode e
JOIN sf_025_titles t
  ON e.tconst = t.tconst
JOIN sf_025_titles p
  ON e.parentTconst = p.tconst
JOIN title_basics bt
  ON e.tconst = bt.tconst
JOIN title_basics bp
  ON e.parentTconst = bp.tconst
""", OUT_025 / "title.episode.tsv")

In [ ]:
# Block 123: Export title.episode.tsv for SF0.5.
export_query_to_tsv(con, """
SELECT e.*
FROM title_episode e
JOIN sf_05_titles t
  ON e.tconst = t.tconst
JOIN sf_05_titles p
  ON e.parentTconst = p.tconst
JOIN title_basics bt
  ON e.tconst = bt.tconst
JOIN title_basics bp
  ON e.parentTconst = bp.tconst
""", OUT_05 / "title.episode.tsv")

In [ ]:
# Block 124: Export title.episode.tsv for SF1.
export_query_to_tsv(con, """
SELECT e.*
FROM title_episode e
JOIN sf_1_titles t
  ON e.tconst = t.tconst
JOIN sf_1_titles p
  ON e.parentTconst = p.tconst
JOIN title_basics bt
  ON e.tconst = bt.tconst
JOIN title_basics bp
  ON e.parentTconst = bp.tconst
""", OUT_1 / "title.episode.tsv")

In [ ]:
# Block 125: Check whether any orphan parent references remain after the repair step.
print("SF0.25 episode_parent_missing =", con.execute("""
    SELECT COUNT(*)
    FROM sf025_title_episode e
    LEFT JOIN sf025_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
""").fetchone()[0])

print("SF0.5 episode_parent_missing =", con.execute("""
    SELECT COUNT(*)
    FROM sf05_title_episode e
    LEFT JOIN sf05_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
""").fetchone()[0])

print("SF1 episode_parent_missing =", con.execute("""
    SELECT COUNT(*)
    FROM sf1_title_episode e
    LEFT JOIN sf1_title_basics b
      ON e.parentTconst = b.tconst
    WHERE e.parentTconst IS NOT NULL
      AND b.tconst IS NULL
""").fetchone()[0])